# Day 32 - Bloom filter, HyperLogLog and Count-Min sketch

Three structures that answer set questions in a **fixed** amount of memory, no
matter how much data goes in. None of them stores the data. Each keeps a few
bits per item and pays for it with a bounded, *one-sided* error:

| structure | question | how it lies |
|---|---|---|
| Bloom filter | have I seen `x`? | may say yes when the answer is no |
| HyperLogLog | how many distinct? | relative error, either direction |
| Count-Min sketch | how often was `x`? | never under-counts, may over-count |

The size of the error is a knob you turn. The *direction* it leans is the
design decision, and it is what decides whether you can use one at all.

In [1]:
import hashlib
import heapq
import math
import random
import sys
from collections import Counter


def hash64(item, seed=0):
    """A stable 64-bit hash.

    Python's built-in hash() is randomised per process for str and bytes
    (PYTHONHASHSEED), so a sketch built in one process would disagree with the
    same sketch built in another.  Every structure on this page needs a hash
    that is a pure function of the bytes, so they all come through here.
    """
    h = hashlib.blake2b(repr(item).encode(), digest_size=16,
                        salt=seed.to_bytes(2, 'little').ljust(16, b'\0')[:16])
    return int.from_bytes(h.digest()[:8], 'little')


def hash_pair(item):
    """Two independent 64-bit hashes out of one digest.

    Kirsch-Mitzenmacher: k hash functions can be simulated with two, using
    g_i(x) = h1(x) + i*h2(x) + i*i, with no measurable loss in false positive
    rate.  One digest instead of k is the whole reason Bloom filters are fast.
    """
    d = hashlib.blake2b(repr(item).encode(), digest_size=16).digest()
    return int.from_bytes(d[:8], 'little'), int.from_bytes(d[8:], 'little')

print(hash64('cat'), hash64('cat'))          # stable across calls
print(hash64('cat', seed=1))                 # a different family
print(hash_pair('cat'))                      # two hashes from one digest

6549593423243429893 6549593423243429893
6276356063019152848
(6549593423243429893, 6548553939403693670)


## 1. Bloom filter

A bit array of `m` bits and `k` hash functions. To add an item, set the `k`
bits it hashes to. To query, check those same `k` bits: if any is 0 the item
was definitely never added; if all are 1 it *probably* was.

The two design formulas:

- `m = -n ln(p) / (ln 2)^2` bits for `n` items at false positive rate `p`
- `k = (m/n) ln 2` hash functions

Instead of `k` separate hash functions we use Kirsch-Mitzenmacher double
hashing: `g_i(x) = h1(x) + i*h2(x) + i^2`, which behaves like `k` independent
hashes but costs one digest.

In [2]:
def optimal_bits(n, fpr):
    """m = -n ln(p) / (ln 2)^2 - the bits needed for n items at rate p."""
    return max(8, int(math.ceil(-n * math.log(fpr) / (math.log(2) ** 2))))


def optimal_hashes(m, n):
    """k = (m/n) ln 2 - too few hashes underuses the bits, too many fills them."""
    return max(1, int(round(m / n * math.log(2))))


class BloomFilter:
    """A bit array plus k hash functions.  Membership, never the members."""

    def __init__(self, n=1000, fpr=0.01, m=None, k=None):
        self.m = m if m is not None else optimal_bits(n, fpr)
        self.k = k if k is not None else optimal_hashes(self.m, n)
        self.bits = bytearray((self.m + 7) // 8)
        self.n = 0

    def indices(self, item):
        h1, h2 = hash_pair(item)
        return [(h1 + i * h2 + i * i) % self.m for i in range(self.k)]

    def add(self, item):
        for i in self.indices(item):
            self.bits[i >> 3] |= 1 << (i & 7)
        self.n += 1

    def __contains__(self, item):
        return all(self.bits[i >> 3] >> (i & 7) & 1 for i in self.indices(item))

    def bits_set(self):
        return sum(bin(b).count('1') for b in self.bits)

    def fill_ratio(self):
        return self.bits_set() / self.m

    def expected_fpr(self):
        """(1 - e^(-kn/m))^k - what the filter promises at its current load."""
        return (1 - math.exp(-self.k * self.n / self.m)) ** self.k

    def estimated_items(self):
        """Count the items back out of the fill ratio - Swamidass & Baldi."""
        f = self.fill_ratio()
        if f >= 1.0:
            return float('inf')
        return -self.m / self.k * math.log(1 - f)

    def nbytes(self):
        return len(self.bits)

bf = BloomFilter(n=1000, fpr=0.01)
print('m =', bf.m, 'bits,  k =', bf.k, 'hashes,', bf.nbytes(), 'bytes')
for w in ['cat', 'dog', 'fox']:
    bf.add(w)
print("indices of 'cat':", bf.indices('cat'))
print("'cat' in bf:", 'cat' in bf, "  'owl' in bf:", 'owl' in bf)

m = 9586 bits,  k = 7 hashes, 1199 bytes
indices of 'cat': [6745, 4656, 2569, 484, 7987, 5906, 3827]
'cat' in bf: True   'owl' in bf: False


In [3]:
def bloom_report(n=10000, fpr=0.01, trials=100000, seed=1):
    """Build a filter at its design load and measure the promise."""
    rnd = random.Random(seed)
    bf = BloomFilter(n=n, fpr=fpr)
    present = ['item-%d' % i for i in range(n)]
    for x in present:
        bf.add(x)
    misses = sum(('absent-%d-%d' % (i, rnd.randrange(1 << 30))) in bf
                 for i in range(trials))
    exact = set(present)
    return {
        'n': n, 'm': bf.m, 'k': bf.k, 'bytes': bf.nbytes(),
        'bits_per_item': bf.m / n,
        'fill': bf.fill_ratio(),
        'design_fpr': fpr,
        'expected_fpr': bf.expected_fpr(),
        'measured_fpr': misses / trials,
        'false_negatives': sum(x not in bf for x in present),
        'estimated_items': bf.estimated_items(),
        'exact_set_bytes': sys.getsizeof(exact)
                           + sum(sys.getsizeof(s) for s in present),
    }

r = bloom_report()
print('%d items at a %.0f%% target:' % (r['n'], 100 * r['design_fpr']))
print('  m = %d bits (%.2f bits/item), k = %d' % (r['m'], r['bits_per_item'], r['k']))
print('  filter %d bytes vs an exact set %d bytes (%.0fx)'
      % (r['bytes'], r['exact_set_bytes'], r['exact_set_bytes'] / r['bytes']))
print('  bits set: %.1f%%' % (100 * r['fill']))
print('  expected %.2f%%  measured %.2f%%'
      % (100 * r['expected_fpr'], 100 * r['measured_fpr']))
print('  false negatives:', r['false_negatives'], '<- always zero')
print('  items estimated from the fill ratio: %.0f' % r['estimated_items'])

10000 items at a 1% target:
  m = 95851 bits (9.59 bits/item), k = 7
  filter 11982 bytes vs an exact set 1103394 bytes (92x)
  bits set: 51.7%
  expected 1.00%  measured 1.03%
  false negatives: 0 <- always zero
  items estimated from the fill ratio: 9977


### The rate is a promise about a *load*, not about the structure

A filter built for 1000 items does not degrade gently when you put 8000 in it.
And `k` is a real optimum: more hash functions give a query more chances to
find a zero, but they also fill the array faster.

In [4]:
def overfill_report(design_n=1000, fpr=0.01, trials=20000, seed=2):
    """The rate is a promise about a load, not about the structure."""
    rnd = random.Random(seed)
    bf = BloomFilter(n=design_n, fpr=fpr)
    rows, added = [], 0
    for mult in (0.5, 1, 2, 4, 8):
        while added < design_n * mult:
            bf.add('x-%d' % added)
            added += 1
        miss = sum(('q-%d-%d' % (i, rnd.randrange(1 << 30))) in bf
                   for i in range(trials))
        rows.append({'load': mult, 'items': added, 'fill': bf.fill_ratio(),
                     'expected': bf.expected_fpr(), 'measured': miss / trials})
    return rows


def k_sweep(n=2000, bits_per_item=10, trials=50000, seed=3):
    """Why k = (m/n) ln 2: too few hashes and too many are both worse."""
    rnd = random.Random(seed)
    m = n * bits_per_item
    best = optimal_hashes(m, n)
    rows = []
    for k in range(1, 13):
        bf = BloomFilter(m=m, k=k)
        for i in range(n):
            bf.add('k-%d' % i)
        miss = sum(('z-%d-%d' % (i, rnd.randrange(1 << 30))) in bf
                   for i in range(trials))
        rows.append({'k': k, 'fill': bf.fill_ratio(),
                     'theory': bf.expected_fpr(), 'measured': miss / trials,
                     'best': k == best})
    return rows, best

print('%6s %8s %8s %11s %11s' % ('load', 'items', 'fill', 'expected', 'measured'))
for row in overfill_report():
    print('%5.1fx %8d %7.1f%% %10.2f%% %10.2f%%'
          % (row['load'], row['items'], 100 * row['fill'],
             100 * row['expected'], 100 * row['measured']))

print()
rows, best = k_sweep()
print('at 10 bits per item:')
for row in rows:
    print('  k = %2d  theory %6.2f%%  measured %6.2f%% %s'
          % (row['k'], 100 * row['theory'], 100 * row['measured'],
             '<- (m/n) ln 2' if row['best'] else ''))

  load    items     fill    expected    measured
  0.5x      500    30.7%       0.03%       0.01%
  1.0x     1000    52.5%       1.00%       1.26%
  2.0x     2000    77.1%      15.74%      16.19%
  4.0x     4000    94.8%      67.86%      69.18%
  8.0x     8000    99.7%      97.99%      97.50%

at 10 bits per item:
  k =  1  theory   9.52%  measured   9.48% 
  k =  2  theory   3.29%  measured   3.23% 
  k =  3  theory   1.74%  measured   1.79% 
  k =  4  theory   1.18%  measured   1.16% 
  k =  5  theory   0.94%  measured   0.90% 
  k =  6  theory   0.84%  measured   0.80% 
  k =  7  theory   0.82%  measured   0.85% <- (m/n) ln 2
  k =  8  theory   0.85%  measured   0.80% 
  k =  9  theory   0.91%  measured   0.90% 
  k = 10  theory   1.02%  measured   0.88% 
  k = 11  theory   1.16%  measured   1.12% 
  k = 12  theory   1.36%  measured   1.24% 


### Why there is no delete

The bits are shared. Clearing the bits of `y` can flip a bit that `x` was
relying on, and the filter's one guarantee - no false negatives - is gone.
A counting Bloom filter replaces each bit with a small counter so a delete can
undo exactly one add, at roughly four times the memory.

In [5]:
class CountingBloom(BloomFilter):
    """Counters instead of bits, so a delete can undo exactly one add."""

    def __init__(self, n=1000, fpr=0.01, m=None, k=None, width=4):
        super().__init__(n=n, fpr=fpr, m=m, k=k)
        self.counters = [0] * self.m
        self.cap = (1 << width) - 1
        self.saturated = 0

    def add(self, item):
        for i in self.indices(item):
            if self.counters[i] < self.cap:
                self.counters[i] += 1
            else:
                self.saturated += 1
        self.n += 1

    def delete(self, item):
        if item not in self:
            return False
        for i in self.indices(item):
            if 0 < self.counters[i] < self.cap:
                self.counters[i] -= 1
        self.n -= 1
        return True

    def __contains__(self, item):
        return all(self.counters[i] for i in self.indices(item))


def deletion_report(seed=4):
    """Clearing the bits of one item can delete a different one.

    A Bloom filter has no delete, and the reason is worth seeing rather than
    being told: the bits are shared, so zeroing the bits of 'y' can flip a bit
    that 'x' was relying on - and the filter's one guarantee, no false
    negatives, is gone.
    """
    words = ['alpha', 'bravo', 'charlie', 'delta', 'echo', 'foxtrot', 'golf',
             'hotel', 'india', 'juliet', 'kilo', 'lima', 'mike', 'november']
    bf = BloomFilter(m=64, k=3)
    for w in words:
        bf.add(w)
    victim = collide = None
    for y in words:
        ys = set(bf.indices(y))
        for x in words:
            if x != y and set(bf.indices(x)) & ys:
                victim, collide = x, y
                break
        if victim:
            break
    for i in bf.indices(collide):                 # the naive "delete"
        bf.bits[i >> 3] &= ~(1 << (i & 7)) & 0xff
    broken = [w for w in words if w != collide and w not in bf]

    cb = CountingBloom(m=64, k=3)
    for w in words:
        cb.add(w)
    cb.delete(collide)
    still = [w for w in words if w != collide and w not in cb]
    return {
        'deleted': collide, 'shared_with': victim,
        'shared_index': sorted(set(bf.indices(victim)) & set(bf.indices(collide))),
        'false_negatives': broken,
        'counting_false_negatives': still,
        'counting_still_finds_deleted': collide in cb,
    }

d = deletion_report()
print("'%s' and '%s' share bit(s) %s"
      % (d['deleted'], d['shared_with'], d['shared_index']))
print("after naively clearing the bits of '%s':" % d['deleted'])
print('  plain Bloom now answers NO for:', d['false_negatives'])
print('  counting Bloom answers NO for:', d['counting_false_negatives'])
print("  and it correctly forgets '%s': %s"
      % (d['deleted'], not d['counting_still_finds_deleted']))

'alpha' and 'bravo' share bit(s) [25]
after naively clearing the bits of 'alpha':
  plain Bloom now answers NO for: ['bravo', 'india']
  counting Bloom answers NO for: []
  and it correctly forgets 'alpha': True


### The misuse worth knowing: dedup

A Bloom filter in front of an expensive lookup is a great idea - a false
positive costs one wasted read. A Bloom filter *as* the deduplicator is a data
loss bug: a false positive means "already seen", so a document that was never
seen is thrown away and never comes back. Same structure, same error rate; the
difference is entirely in what the code after the answer does with a wrong
one.

In [6]:
def dedup_report(docs=20000, dup_rate=0.3, fpr=0.01, seed=5):
    """Dedup is the misuse case: a false positive silently drops a document."""
    rnd = random.Random(seed)
    stream, uniq = [], 0
    for i in range(docs):
        if stream and rnd.random() < dup_rate:
            stream.append(rnd.choice(stream))
        else:
            stream.append('doc-%d' % uniq)
            uniq += 1
    bf = BloomFilter(n=uniq, fpr=fpr)
    kept_bloom, seen = [], set()
    for d in stream:
        if d not in bf:
            bf.add(d)
            kept_bloom.append(d)
    kept_exact = [d for d in stream if not (d in seen or seen.add(d))]
    lost = set(kept_exact) - set(kept_bloom)
    return {
        'documents': docs, 'unique': uniq,
        'kept_exact': len(kept_exact), 'kept_bloom': len(kept_bloom),
        'lost': len(lost), 'lost_pct': 100 * len(lost) / uniq,
        'fpr': fpr, 'bloom_bytes': bf.nbytes(),
        'exact_bytes': sys.getsizeof(seen) + sum(sys.getsizeof(s) for s in seen),
    }

g = dedup_report()
print('%d documents, %d distinct, filter designed for %.0f%%'
      % (g['documents'], g['unique'], 100 * g['fpr']))
print('  exact dedup keeps %d' % g['kept_exact'])
print('  bloom dedup keeps %d' % g['kept_bloom'])
print('  distinct documents lost: %d (%.2f%%)' % (g['lost'], g['lost_pct']))

20000 documents, 13954 distinct, filter designed for 1%
  exact dedup keeps 13954
  bloom dedup keeps 13932
  distinct documents lost: 22 (0.16%)


## 2. HyperLogLog

Counting distinct items exactly costs one entry per item. HyperLogLog costs
16 KB, forever.

The idea is that a *maximum* can measure a count. If you flip a coin until
tails, a run of `r` heads shows up about once in `2^r` tries - so the longest
run you have ever seen estimates `log2(how many times you tried)`, and a
maximum needs one number of storage rather than a list.

One hash does two jobs: the low `p` bits pick one of `m = 2^p` registers, and
the run of zeros in the remaining bits is that item's `rho`. Each register
keeps the largest `rho` it has seen. The estimate is the harmonic mean of
`2^register` - harmonic, because one lucky item with 20 zeros would dominate
an arithmetic mean and barely moves a harmonic one.

In [7]:
def leading_zeros_intuition(trials=2000, seed=7):
    """Why a maximum can count: flip coins, remember the longest head run.

    If n people each flip until tails, the longest run of heads seen is about
    log2(n).  So 2**(longest run) is an estimate of n - built from one number,
    not from a list of who flipped what.
    """
    rnd = random.Random(seed)
    rows = []
    for n in (8, 64, 512, 4096):
        runs = []
        for _ in range(trials // 8):
            longest = 0
            for _ in range(n):
                r = 0
                while rnd.random() < 0.5:
                    r += 1
                longest = max(longest, r)
            runs.append(longest)
        avg = sum(runs) / len(runs)
        rows.append({'n': n, 'log2n': math.log2(n), 'avg_longest_run': avg,
                     'estimate': 2 ** avg})
    return rows

print('%6s %9s %17s %10s' % ('n', 'log2(n)', 'avg longest run', '2**run'))
for row in leading_zeros_intuition():
    print('%6d %9.2f %17.2f %10.0f'
          % (row['n'], row['log2n'], row['avg_longest_run'], row['estimate']))

     n   log2(n)   avg longest run     2**run
     8      3.00              3.47         11
    64      6.00              6.25         76
   512      9.00              9.29        627
  4096     12.00             12.20       4718


In [8]:
class HyperLogLog:
    """m registers, each holding the longest run of leading zeros it has seen.

    The estimate is the harmonic mean of 2**register over all registers, which
    is what tames the variance: one lucky item with 20 leading zeros would
    dominate an arithmetic mean, but barely moves a harmonic one.
    """

    def __init__(self, p=14):
        self.p = p
        self.m = 1 << p
        self.reg = bytearray(self.m)
        self.alpha = {4: 0.673, 5: 0.697, 6: 0.709}.get(
            p, 0.7213 / (1 + 1.079 / self.m))

    def add(self, item):
        x = hash64(item)
        idx = x & (self.m - 1)                    # which register
        w = x >> self.p                           # the rest decides the run
        rho = 1
        while rho <= 64 - self.p and not (w >> (rho - 1)) & 1:
            rho += 1
        if rho > self.reg[idx]:
            self.reg[idx] = rho

    def count(self):
        z = sum(2.0 ** -r for r in self.reg)
        est = self.alpha * self.m * self.m / z
        zeros = self.reg.count(0)
        if est <= 2.5 * self.m and zeros:
            return self.m * math.log(self.m / zeros)   # linear counting
        return est

    def merge(self, other):
        """Union of two sets, from the two sketches, with no re-reading.

        Register-wise max, and that is the whole operation.  It is exact in the
        sense that the merged sketch equals the sketch you would have built by
        feeding it both streams - which is why HLLs can be summed across
        shards, days, or machines.
        """
        assert self.p == other.p
        out = HyperLogLog(self.p)
        out.reg = bytearray(max(a, b) for a, b in zip(self.reg, other.reg))
        return out

    def nbytes(self):
        return self.m


def hll_report(p=14, sizes=(1000, 10000, 100000, 500000), seed=8):
    hll_bytes = 1 << p
    rows = []
    for n in sizes:
        h = HyperLogLog(p)
        for i in range(n):
            h.add('u-%d-%d' % (seed, i))
        est = h.count()
        exact = sys.getsizeof(set(range(n))) + 32 * n   # 32B per small str, low
        rows.append({'n': n, 'estimate': est,
                     'error_pct': 100 * (est - n) / n,
                     'hll_bytes': hll_bytes, 'exact_bytes': exact,
                     'ratio': exact / hll_bytes})
    return rows, 1.04 / math.sqrt(1 << p) * 100

rows, theo = hll_report(sizes=(1000, 10000, 100000))
print('p = 14 -> 16384 registers, theory says 1.04/sqrt(m) = %.2f%%' % theo)
print('%9s %11s %9s %10s %12s' % ('true', 'estimate', 'error', 'hll bytes', 'exact bytes'))
for row in rows:
    print('%9d %11.0f %8.2f%% %10d %12d'
          % (row['n'], row['estimate'], row['error_pct'],
             row['hll_bytes'], row['exact_bytes']))

p = 14 -> 16384 registers, theory says 1.04/sqrt(m) = 0.81%
     true    estimate     error  hll bytes  exact bytes
     1000        1005     0.52%      16384        64984
    10000        9930    -0.70%      16384       844504
   100000       99924    -0.08%      16384      7394520


### Union is free, intersection is not

Merging two sketches is a register-wise max, and the result is *exactly* the
sketch you would have built from both streams. That is why HLL is the one
that actually ships: every shard keeps 16 KB and the rollup is a max.

Inclusion-exclusion, on the other hand, subtracts three large numbers to get a
small one. Each carries about 0.8% *relative* error, and after the subtraction
those become one *absolute* error the size of the answer.

In [9]:
def hll_intersection_report(p=14, n=200000, overlap=2000, seed=9):
    """Union is free; intersection is where the error eats the answer.

    |A and B| = |A| + |B| - |A or B| subtracts three numbers each carrying
    ~0.8% relative error.  When the sets are big and the overlap is small, the
    absolute errors are the same size as the answer.
    """
    a, b = HyperLogLog(p), HyperLogLog(p)
    for i in range(n):
        a.add('a-%d' % i)
    for i in range(n - overlap, 2 * n - overlap):
        b.add('a-%d' % i if i < n else 'b-%d' % i)
    ca, cb = a.count(), b.count()
    cu = a.merge(b).count()
    inter = ca + cb - cu
    return {'n': n, 'true_union': 2 * n - overlap, 'true_intersection': overlap,
            'est_a': ca, 'est_b': cb, 'est_union': cu,
            'union_error_pct': 100 * (cu - (2 * n - overlap)) / (2 * n - overlap),
            'est_intersection': inter,
            'intersection_error_pct': 100 * (inter - overlap) / overlap}

a, b, whole = HyperLogLog(14), HyperLogLog(14), HyperLogLog(14)
for i in range(50000):
    a.add('x-%d' % i)
    whole.add('x-%d' % i)
for i in range(40000, 90000):
    b.add('x-%d' % i)
    whole.add('x-%d' % i)
merged = a.merge(b)
print('true union      %d' % 90000)
print('merged sketches %.0f' % merged.count())
print('single pass     %.0f' % whole.count())
print('registers identical:', merged.reg == whole.reg)

print()
i = hll_intersection_report()
print('|A| = |B| = %d, true intersection %d' % (i['n'], i['true_intersection']))
print('  union estimate off by %+.2f%%' % i['union_error_pct'])
print('  intersection estimate %.0f, off by %+.0f%%'
      % (i['est_intersection'], i['intersection_error_pct']))

true union      90000
merged sketches 90647
single pass     90647
registers identical: True

|A| = |B| = 200000, true intersection 2000
  union estimate off by -0.84%
  intersection estimate 1681, off by -16%


## 3. Count-Min sketch

`d` rows of `w` counters. Every row counts the *whole* stream, so every row is
an over-estimate of any particular key; the answer is the minimum, which picks
the row that suffered the fewest collisions.

With `w = ceil(e/eps)` and `d = ceil(ln(1/delta))`, the estimate is at most
`truth + eps*N` with probability `1 - delta`. Note the shape of that bound: the
error is *absolute*, proportional to the total stream. On a heavy hitter it is
rounding noise; on a key that appeared once it can be the entire answer. Which
is exactly the right trade when the question is "which keys are big?".

In [10]:
class CountMinSketch:
    """d rows of w counters; every row is a complete, lossy count of the stream.

    Every row counts *everything* - the whole stream is added to every row -
    so every row is an over-estimate, and taking the minimum picks the row that
    happened to give this item the fewest collisions.
    """

    def __init__(self, epsilon=0.001, delta=0.01, w=None, d=None,
                 conservative=False):
        self.w = w if w is not None else int(math.ceil(math.e / epsilon))
        self.d = d if d is not None else int(math.ceil(math.log(1 / delta)))
        self.rows = [[0] * self.w for _ in range(self.d)]
        self.conservative = conservative
        self.total = 0

    def _cols(self, item):
        h1, h2 = hash_pair(item)
        return [(h1 + r * h2 + r * r) % self.w for r in range(self.d)]

    def add(self, item, count=1):
        cols = self._cols(item)
        if self.conservative:
            # only raise the counters that are currently at the minimum -
            # the others already over-estimate, so raising them adds error
            cur = min(self.rows[r][c] for r, c in enumerate(cols))
            for r, c in enumerate(cols):
                self.rows[r][c] = max(self.rows[r][c], cur + count)
        else:
            for r, c in enumerate(cols):
                self.rows[r][c] += count
        self.total += count

    def query(self, item):
        return min(self.rows[r][c] for r, c in enumerate(self._cols(item)))

    def nbytes(self):
        return self.w * self.d * 4                 # 32-bit counters

    def error_bound(self):
        """The guarantee: estimate <= truth + eps*N, with probability 1-delta."""
        return math.e / self.w * self.total


def zipf_stream(n_items=5000, length=200000, skew=1.1, seed=11):
    rnd = random.Random(seed)
    weights = [1.0 / (i + 1) ** skew for i in range(n_items)]
    total = sum(weights)
    cum, acc = [], 0.0
    for w in weights:
        acc += w / total
        cum.append(acc)
    from bisect import bisect_left
    return ['w-%d' % bisect_left(cum, rnd.random()) for _ in range(length)]


def cms_report(stream=None, epsilon=0.0005, delta=0.01, seed=11):
    stream = stream if stream is not None else zipf_stream(seed=seed)
    truth = Counter(stream)
    plain = CountMinSketch(epsilon, delta)
    cons = CountMinSketch(epsilon, delta, conservative=True)
    for x in stream:
        plain.add(x)
        cons.add(x)
    items = sorted(truth, key=lambda k: -truth[k])
    heavy, tail = items[:10], items[-10:]
    err = [plain.query(k) - truth[k] for k in truth]
    under = sum(plain.query(k) < truth[k] for k in truth)
    return {
        'distinct': len(truth), 'length': len(stream),
        'w': plain.w, 'd': plain.d, 'bytes': plain.nbytes(),
        'exact_bytes': sys.getsizeof(truth) + 60 * len(truth),
        'bound': plain.error_bound(),
        'max_error': max(err), 'mean_error': sum(err) / len(err),
        'under_counts': under,
        'heavy': [(k, truth[k], plain.query(k), cons.query(k)) for k in heavy[:5]],
        'tail': [(k, truth[k], plain.query(k), cons.query(k)) for k in tail[:5]],
        'heavy_rel': 100 * sum(plain.query(k) / truth[k] - 1 for k in heavy) / 10,
        'tail_rel': 100 * sum(plain.query(k) / truth[k] - 1 for k in tail) / 10,
        'cons_mean_error': sum(cons.query(k) - truth[k] for k in truth) / len(truth),
    }

c = cms_report()
print('%d distinct keys, %d events -> %d rows x %d counters = %d bytes'
      % (c['distinct'], c['length'], c['d'], c['w'], c['bytes']))
print('an exact Counter would be about %d bytes' % c['exact_bytes'])
print('guaranteed bound e/w * N = %.0f, worst observed error %d, under-counts %d'
      % (c['bound'], c['max_error'], c['under_counts']))
print()
print('%10s %9s %9s %14s' % ('key', 'true', 'CMS', 'conservative'))
for k, t, q, cq in c['heavy']:
    print('%10s %9d %9d %14d' % (k, t, q, cq))
print('   ...the tail...')
for k, t, q, cq in c['tail']:
    print('%10s %9d %9d %14d' % (k, t, q, cq))
print('relative error, top 10: %+.2f%%   bottom 10: %+.0f%%'
      % (c['heavy_rel'], c['tail_rel']))

4934 distinct keys, 200000 events -> 5 rows x 5437 counters = 108740 bytes
an exact Counter would be about 443608 bytes
guaranteed bound e/w * N = 100, worst observed error 21, under-counts 0

       key      true       CMS   conservative
       w-0     31648     31648          31648
       w-1     14717     14717          14717
       w-2      9415      9415           9415
       w-3      6847      6847           6847
       w-4      5464      5464           5464
   ...the tail...
    w-4347         1         1              1
    w-4054         1         1              1
    w-4841         1         4              3
    w-4561         1         9              9
    w-4263         1         5              4
relative error, top 10: +0.01%   bottom 10: +150%


## LeetCode 347 - Top K Frequent Elements

Given an array and `k`, return the `k` most frequent elements. The accepted
answer is exact: count everything, then a `k`-sized heap. `O(n)` time,
`O(distinct)` space.

That second term is the one that stops being affordable when `nums` is not an
array but a firehose. A Count-Min sketch plus a `k`-sized heap answers the
same question in memory that does not depend on how many distinct keys go by.
Because the sketch never under-counts, a genuine heavy hitter can never be
pushed out by a lighter one - what *can* happen is that a rare key gets
inflated enough to sneak in. So the approximation is only meaningful when the
head stands clear of the noise floor.

In [11]:
def top_k_frequent(nums, k):
    """LC 347, the exact answer: count everything, then a k-sized heap.

    O(n) time and O(distinct) space - and that second term is the one that
    stops being affordable when 'nums' is a firehose rather than a list.
    """
    freq = Counter(nums)
    return [x for x, _ in heapq.nlargest(k, freq.items(), key=lambda p: p[1])]


def top_k_streaming(stream, k, epsilon=0.0005, delta=0.01, conservative=True):
    """The same question with a sketch plus a k-sized heap of candidates.

    Memory is w*d counters plus k entries, independent of how many distinct
    items go by.  The sketch never under-counts, so a genuine heavy hitter can
    never be pushed out by a lighter one - what can happen is that a rare item
    is inflated enough to sneak in.
    """
    cms = CountMinSketch(epsilon, delta, conservative=conservative)
    heap, inside = [], {}
    for x in stream:
        cms.add(x)
        c = cms.query(x)
        if x in inside:
            inside[x] = c
        elif len(heap) < k:
            inside[x] = c
        else:
            lo = min(inside, key=lambda t: inside[t])
            if c > inside[lo]:
                del inside[lo]
                inside[x] = c
    heap = sorted(inside, key=lambda t: -inside[t])
    return heap[:k], cms


def lc347_report(k=10, seed=11):
    stream = zipf_stream(seed=seed)
    exact = top_k_frequent(stream, k)
    approx, cms = top_k_streaming(stream, k)
    truth = Counter(stream)
    flat = zipf_stream(n_items=5000, length=200000, skew=0.05, seed=seed)
    exact_f = top_k_frequent(flat, k)
    approx_f, _ = top_k_streaming(flat, k)
    tf = Counter(flat)
    return {
        'k': k,
        'exact': exact, 'approx': approx,
        'same_set': set(exact) == set(approx),
        'same_order': exact == approx,
        'exact_counts': [truth[x] for x in exact],
        'approx_counts': [cms.query(x) for x in approx],
        'cms_bytes': cms.nbytes(),
        'counter_bytes': sys.getsizeof(truth) + 60 * len(truth),
        'flat_same_set': set(exact_f) == set(approx_f),
        'flat_gap': tf[exact_f[0]] - tf[exact_f[-1]],
        'flat_top': tf[exact_f[0]], 'flat_kth': tf[exact_f[k - 1]],
        'flat_missed': sorted(set(exact_f) - set(approx_f)),
    }

print(top_k_frequent([1, 1, 1, 2, 2, 3], 2))
print(top_k_frequent([1], 1))

lc = lc347_report()
print()
print('200k-event Zipf stream, k = %d' % lc['k'])
print('  exact  %s' % lc['exact'])
print('  sketch %s' % lc['approx'])
print('  same set: %s, same order: %s' % (lc['same_set'], lc['same_order']))
print('  sketch %d bytes vs Counter %d bytes' % (lc['cms_bytes'], lc['counter_bytes']))
print()
print('a nearly flat stream (top %d vs kth %d):' % (lc['flat_top'], lc['flat_kth']))
print('  same set: %s, missed: %s' % (lc['flat_same_set'], lc['flat_missed']))

[1, 2]
[1]

200k-event Zipf stream, k = 10
  exact  ['w-0', 'w-1', 'w-2', 'w-3', 'w-4', 'w-5', 'w-6', 'w-7', 'w-8', 'w-9']
  sketch ['w-0', 'w-1', 'w-2', 'w-3', 'w-4', 'w-5', 'w-6', 'w-7', 'w-8', 'w-9']
  same set: True, same order: True
  sketch 108740 bytes vs Counter 443608 bytes

a nearly flat stream (top 65 vs kth 61):
  same set: False, missed: ['w-385']


## Picking one

| structure | question | how it lies | mergeable |
|---|---|---|---|
| Bloom filter | have I seen `x`? | false positive only | yes, OR the bit arrays |
| HyperLogLog | how many distinct? | relative, both directions | yes, max the registers |
| Count-Min | how often is `x`? | over-count only | yes, add the rows |

Bloom filters sit in front of expensive lookups (Chrome Safe Browsing,
Cassandra and LevelDB SSTables). HyperLogLog is what
`APPROX_COUNT_DISTINCT` and Redis `PFCOUNT` are. Count-Min shows up in flow
monitoring, trending topics, and per-key rate limiting.

The habit worth taking away: before using any of them, ask what the code does
with a wrong answer, and check that the error can only lean in a direction
that code survives.

In [12]:
assert r['false_negatives'] == 0
assert r['measured_fpr'] < 3 * r['design_fpr']
assert len(d['false_negatives']) > 0
assert len(d['counting_false_negatives']) == 0
assert g['lost'] > 0
assert all(abs(row['error_pct']) < 2.5 for row in rows)
assert merged.reg == whole.reg
assert abs(i['intersection_error_pct']) > 10
assert c['under_counts'] == 0
assert c['max_error'] <= c['bound']
assert top_k_frequent([1, 1, 1, 2, 2, 3], 2) == [1, 2]
assert lc['same_set']
print('all assertions passed')

all assertions passed
